## Configuration and Imports
Set up the Ollama API connection and define the models to use for chat and embeddings.

**Key Definitions:**
- `HOST`: Base URL for Ollama API (localhost:11434)
- `MODEL_NAME`: LLM model for chat and reasoning (netops-llama3.2-vision)
- `EMBED_MODEL_NAME`: Embedding model for vectorizing text (embeddinggemma)

In [1]:
import json, urllib.request, urllib.error, os, math, re
from dataclasses import dataclass
from typing import List, Tuple, Dict

HOST = "http://localhost:11434"

MODEL_NAME = "netops-llama3.2-vision:latest"
EMBED_MODEL_NAME = "embeddinggemma:latest"

def ollama_get(path: str, timeout=10) -> dict:
    request = urllib.request.Request(HOST + path, method="GET")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))
    
print("Ollama version:", ollama_get("/api/version"))
tags = ollama_get("/api/tags")
print("Available models:", [m["name"] for m in tags.get("models", [])][:10], "...")


Ollama version: {'version': '0.17.7'}
Available models: ['netops-llama3.2-vision:latest', 'gemma:7b', 'embeddinggemma:latest', 'llama3.2-vision:latest', 'llama3.1:8b', 'gpt-oss:20b'] ...


## API Functions for Ollama
Define utility functions to communicate with the Ollama API for embeddings and chat.

**Key Functions:**
- `ollama_post()`: Send POST requests to Ollama API with JSON payloads
- `embed_text()`: Convert text into dense vector embeddings for semantic search
- `chat_user_only()`: Send a user message to the LLM and receive a response with temperature control

In [3]:
def ollama_post(path: str, payload: dict, timeout=120) -> dict:
    request = urllib.request.Request(
        HOST + path,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))
    
def embed_text(text: str) -> List[float]:
    response = ollama_post("/api/embed", {"model": EMBED_MODEL_NAME, "input": text}, timeout=120)
    return response.get("embedding", [])

def chat_user_only(user_content: str, temperature: float = 0.2) -> str:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": user_content}],
        "stream": False,
        "options": {"temperature": temperature}
        }
    response = ollama_post("/api/chat", payload, timeout=180)
    return response.get("message", {}).get("content", "")

## Data Redaction Configuration
Define patterns and rules for automatically redacting sensitive information from network operations documents.

**Key Components:**
- `RedactionConfig`: Configuration dataclass controlling which types of data to redact (IPs, MACs, emails, phone numbers, etc.)
- `redact_text()`: Main function that applies regex patterns to replace sensitive data with labeled tokens (e.g., `<REDACTED:IPV4>`)
- **Redaction Patterns**: Regex patterns for IPv4/IPv6 addresses, MAC addresses, network interfaces, email addresses, phone numbers, and physical addresses
- `CFG`: Global configuration instance with default redaction settings

**Purpose:**
Protect sensitive information in operational documents, logs, and runbooks before embedding and indexing them for RAG retrieval, ensuring compliance and security while maintaining document context and usefulness.

In [4]:
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple, Pattern, Optional

@dataclass
class RedactionConfig:
    redact_ipv4: bool = True
    redact_ipv6: bool = True
    redact_mac: bool = True
    redact_interfaces: bool = True
    redact_phone: bool = True
    redact_email: bool = True
    redact_addresses: bool = False  # heuristic can be aggressive; enable if needed
    customer_names: Optional[List[str]] = None  # exact or regex patterns
    replacement_style: str = "token"  # "token" or "hash"

CFG = RedactionConfig(
    redact_addresses=False,
    customer_names=[
        # Put internal/customer names or regex patterns here if you want:
        # r"\bAcme\s+Corp\b",
        # r"\bContoso\b",
    ],
    replacement_style="token"
)

# --- Patterns ---
IPV4 = re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b")
# IPv6: decent heuristic; not perfect but practical for logs
IPV6 = re.compile(r"\b(?:[0-9a-fA-F]{1,4}:){2,7}[0-9a-fA-F]{1,4}\b")
MAC  = re.compile(r"\b(?:[0-9A-Fa-f]{2}[:-]){5}[0-9A-Fa-f]{2}\b|\b(?:[0-9A-Fa-f]{4}\.){2}[0-9A-Fa-f]{4}\b")

EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")

# Phone: handles formats like (303) 555-1212, 303-555-1212, +1 303 555 1212
PHONE = re.compile(r"\b(?:\+?1[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]?\d{3}[\s.-]?\d{4}\b")

# Interfaces (heuristic): Gi0/1, Te1/0/1, Eth1/2, xe-0/0/0, ge-0/0/1, et-0/0/0
IFACE = re.compile(r"\b(?:(?:Gi|Te|Fa|Eth|Po|Lo|Vlan)\d+(?:/\d+){0,3}|(?:xe|ge|et)-\d+/\d+/\d+)\b", re.IGNORECASE)

# Addresses (heuristic): "123 Main St", "4567 W Elm Avenue"
ADDR = re.compile(r"\b\d{1,6}\s+(?:[A-Za-z0-9.\-]+\s+){0,6}"
                r"(?:St|Street|Ave|Avenue|Rd|Road|Blvd|Boulevard|Ln|Lane|Dr|Drive|Ct|Court|Way|Pkwy|Parkway)\b",
                re.IGNORECASE)

def _make_replacer(label: str):
    # Simple token replacement (stable label)
    def repl(_m):
        return f"<REDACTED:{label}>"
    return repl

def redact_text(text: str, cfg: RedactionConfig = CFG) -> Tuple[str, Dict[str, int]]:
    """
    Returns (redacted_text, stats).
    NOTE: This is best-effort. It will not catch everything (especially names).
    """
    stats: Dict[str, int] = {}

    def apply(pattern: Pattern, label: str, enabled: bool):
        nonlocal text
        if not enabled:
            return
        before = text
        text, n = pattern.subn(_make_replacer(label), text)
        if n:
            stats[label] = stats.get(label, 0) + n

    apply(IPV4, "IPV4", cfg.redact_ipv4)
    apply(IPV6, "IPV6", cfg.redact_ipv6)
    apply(MAC,  "MAC",  cfg.redact_mac)
    apply(IFACE,"IFACE",cfg.redact_interfaces)
    apply(PHONE,"PHONE",cfg.redact_phone)
    apply(EMAIL,"EMAIL",cfg.redact_email)
    apply(ADDR, "ADDRESS", cfg.redact_addresses)

    # Customer names: safest is explicit patterns/list
    if cfg.customer_names:
        for i, pat in enumerate(cfg.customer_names, start=1):
            p = re.compile(pat, re.IGNORECASE)
            text, n = p.subn(_make_replacer(f"CUSTOMER{i}"), text)
            if n:
                stats["CUSTOMER"] = stats.get("CUSTOMER", 0) + n

    return text, stats

## Text Processing and Chunking
Prepare documents for embedding by normalizing text and splitting into manageable chunks.

**Key Functions:**
- `normalize_text()`: Clean up whitespace and normalize text formatting
- `chunk_text()`: Split documents into overlapping chunks to preserve context (size=1000 chars with 200 char overlap)

**Purpose:**
Transform raw documents into consistent, embedding-friendly chunks that can be efficiently vectorized and retrieved. The overlap between chunks ensures that context isn't lost at chunk boundaries, improving retrieval quality for questions that might span multiple segments of text. This chunking strategy balances between capturing sufficient context in each chunk while keeping embeddings computationally efficient.

In [5]:
def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text.strip())

def chunk_text(text: str, chunk_size: int = 1000, overlap: int=200) -> List[str]:
    """
    Chunk by characters with overlap (simple + reliable).
    For docs/runbooks/logs, we can chunk by lines or sections instead, but for general text this is more robust.
    """
    text = normalize_text(text)
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

## Load and Redact Source Documents
Read documents from the assets/documents directory (supports .txt, .md, .log, .csv, .jsonl, .yml) and apply redaction to protect sensitive information.

**Key Outputs:**
- `DOCUMENTS`: List of redacted document contents loaded from files
- `DOCUMENT_SOURCES`: List of file paths corresponding to each document (for source attribution)
- `DOCUMENT_REDACTION_SUMMARIES`: Dictionary mapping file paths to redaction statistics (counts of redacted items per document)

**Redaction Process:**
- Automatically redacts sensitive data: IPv4/IPv6 addresses, MAC addresses, interface names, phone numbers, email addresses
- Optionally redacts physical addresses and customer names (configurable via `RedactionConfig`)
- Replacement tokens are used (e.g., `<REDACTED:IPV4>`) to maintain context while protecting privacy

In [6]:
import pathlib

DOCUMENTS_DIR = pathlib.Path("../assets/documents")
DOCUMENT_EXTENSIONS = [".txt", ".md", ".log", ".csv", ".jsonl", ".yml"]  # You can add more extensions if needed
DOCUMENT_PATH = [p for p in DOCUMENTS_DIR.glob("*") if p.is_file() and p.suffix.lower() in DOCUMENT_EXTENSIONS]
DOCUMENTS = []
DOCUMENT_SOURCES = []
DOCUMENT_REDACTION_SUMMARIES = {}


for doc_path in sorted(DOCUMENT_PATH):
    try:
        text_raw = doc_path.read_text(encoding="utf-8", errors="replace")
        redacted_text, redaction_stats = redact_text(text_raw, CFG)
        if redacted_text.strip():  # Only include non-empty documents
            DOCUMENTS.append(redacted_text)
            DOCUMENT_SOURCES.append(str(doc_path))
            DOCUMENT_REDACTION_SUMMARIES[str(doc_path)] = redaction_stats
    except Exception as e:
        print(f"Error reading {doc_path}: {e}")

print(f"Loaded {len(DOCUMENTS)} documents from {DOCUMENTS_DIR}/")
print("Sample document sources:", DOCUMENT_SOURCES[:5], "...")
print("Sample redaction summaries:", list(DOCUMENT_REDACTION_SUMMARIES.items())[:5], "...")


# For demonstration, we can embed some sample documents (e.g. runbook sections, troubleshooting checklists) and store them in-memory.
# If you don't currently have any available documents you want to add, you can use the following sample text snippets (these are just examples and can be replaced with your actual documents),
# Just uncomment the below block and comment out the above code that loads documents from the directory, and it will use these sample documents instead of loading from the directory.
#
#  DOCUMENTS = [
# """
# Title: BGP Neighbor Down - Core Checklist
# - Confirm physical link status and errors (interfaces counters).
# - Check BGP hold timers and keepalives; 'hold time expired' often indicates loss/congestion.
# - Verify MTU on the path if sessions flap during peak.
# - Correlate with interface link events (down/up).
# - Validate reachability (ping, traceroute) and control-plane policing.
# """,
# """
# Title: One-Way Audio Troubleshooting (SIP/RTP)
# - Confirm SDP offers correct IP/port; check NAT and ALG behavior.
# - Verify RTP is flowing both directions (packet captures, SBC stats).
# - Look for codec mismatch, asymmetric routing, firewall pinholes timing out.
# - For intermittent issues, correlate with jitter/loss; enable RTCP stats if available.
# """,
# """
# Title: QoS Quick Notes
# - For voice, prioritize RTP (DSCP EF) and SIP signaling (CS3/AF31 per policy).
# - Validate marking at ingress and no remarking in transit.
# - Check queue drops and shaping policies on egress.
# """
# ]


Loaded 4 documents from ../assets/documents/
Sample document sources: ['../assets/documents/bgp_training_data[86].jsonl', '../assets/documents/sample_bgp_mop.txt', '../assets/documents/voip_config_2023.txt', '../assets/documents/voip_config_2026.txt'] ...
Sample redaction summaries: [('../assets/documents/bgp_training_data[86].jsonl', {'IPV4': 60}), ('../assets/documents/sample_bgp_mop.txt', {}), ('../assets/documents/voip_config_2023.txt', {'IPV4': 4}), ('../assets/documents/voip_config_2026.txt', {'IPV4': 4})] ...


## Build Vector Index
Create embeddings for all document chunks and store them in an in-memory vector index for semantic search.

**Key Components:**
- `DocumentChunk`: Data class storing document metadata, source path, text, and embeddings
- `cosine_similarity()`: Compute similarity between query and stored embeddings
- `index`: In-memory vector store with embeddings for all chunks

**Purpose:**
Process all redacted documents by chunking them (700 chars with 120 char overlap) and generating embeddings for each chunk. The embeddings are created from the redacted text, ensuring that sensitive information is not present in the vector representations. Each chunk maintains a reference to its source document for attribution in responses.

In [7]:
from dataclasses import dataclass
from typing import List

@dataclass
class DocumentChunk:
    source: str
    doc_id: int
    chunk_id: int
    text: str
    embedding: List[float]

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot_product = sum(x*y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x*x for x in a))
    norm_b = math.sqrt(sum(y*y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

index: List[DocumentChunk] = []

for doc_id, doc in enumerate(DOCUMENTS):
    chunks = chunk_text(doc, chunk_size=700, overlap=120)
    for chunk_id, chunk in enumerate(chunks):
        embedding = embed_text(chunk)
        index.append(DocumentChunk(source=DOCUMENT_SOURCES[doc_id], doc_id=doc_id, chunk_id=chunk_id, text=chunk, embedding=embedding))

print(f"Indexed {len(DOCUMENTS)} docs in {len(index)} chunks with embeddings model '{EMBED_MODEL_NAME}'.")

Indexed 4 docs in 18 chunks with embeddings model 'embeddinggemma:latest'.


## Semantic Search and Retrieval
Find the most relevant document chunks for a given query using embedding similarity.

**Key Function:**
- `retrieve_relevant_chunks()`: Embed the query and find top-k most similar chunks from the index using cosine similarity scoring

**Purpose:**
Performs semantic search over the redacted document chunks to find the most relevant context for answering questions. The retrieval works on vector embeddings, meaning it can find semantically similar content even when the query uses different wording than the source documents, all while maintaining privacy through the redacted content.

In [8]:
def retrieve_relevant_chunks(query: str, top_k: int = 3) -> List[Tuple[DocumentChunk, float]]:
    query_embedding = embed_text(query)
    scored_chunks = []
    for chunk in index:
        score = cosine_similarity(query_embedding, chunk.embedding)
        scored_chunks.append((chunk, score))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    return scored_chunks[:top_k]

test = retrieve_relevant_chunks("What are the common causes for BGP neighbor down and how to troubleshoot?", top_k=2)
for chunk, score in test:
    print(f"Score: {score:.4f}, DocID: {chunk.doc_id}, ChunkID: {chunk.chunk_id}, Text: {chunk.text[:100]}...")

Score: 0.0000, DocID: 0, ChunkID: 0, Text: {"messages": [{"role": "system", "content": "You are a network engineer. Always use Loopback0 for BG...
Score: 0.0000, DocID: 0, ChunkID: 1, Text: a network engineer. Always use AS 65001"}]} {"messages": [{"role": "user", "content": "Set up BGP fo...


## Generate RAG Response
Combine retrieved context with the user's question and prompt the LLM for a structured answer.

**Key Function:**
- `generate_rag_response()`: Redacts the user's question, retrieves relevant chunks using the redacted query, formats them as context, and sends to the LLM with a structured prompt template to generate answers with bullet points, next steps, verification steps, and source attribution

**Purpose:**
Implements privacy-aware RAG by redacting sensitive information in both the user's question and the document corpus. This ensures that sensitive data (IPs, MACs, emails, etc.) is consistently protected throughout the retrieval and response generation process, while maintaining the semantic meaning needed for accurate answers.

In [9]:
RAG_PROMPT_TEMPLATE = """
Use only the following retrieved document chunks to answer the question. If the retrieved information is insufficient to answer the question, say you don't know. Always use all relevant information from the retrieved chunks in your answer and suggest next steps if applicable.
Return:

1) A concise answer in bullet points to the question based on the retrieved information.
2) Next steps or suggestions for further investigation (Ordered list).
3) Verification steps to confirm the issue or validate the solution.
4) Sources of the retrieved information (list the document sources corresponding to the retrieved chunks).

Keep it concise and focused on the question. Do not include any information that is not directly relevant to the question.
"""

def generate_rag_response(question: str, top_k: int = 4) -> str:
    question_redacted, redaction_stats = redact_text(question, CFG)
    top_chunks = retrieve_relevant_chunks(question_redacted, top_k=top_k)
    context_block = []
    for rec, score in top_chunks:
        context_block.append(
            f"[{rec.doc_id}-{rec.chunk_id}] (score: {score:.4f})\n(Source: {DOCUMENT_SOURCES[rec.doc_id]}\n{rec.text}\n)"
        )
    prompt = f"""Question: {question_redacted}
Retrieved Context (redacted):
{'\n\n'.join(context_block)}

{RAG_PROMPT_TEMPLATE}
"""
    response = chat_user_only(prompt, temperature=0.2)
    return response

# print(generate_rag_response("What are the common causes for BGP neighbor down and how to troubleshoot?", top_k=4))
print(generate_rag_response("BGP neighbor down with high CPU usage and memory utilization events. What could be the cause and what should I do first? Please provide a detailed analysis including the IP address of the affected neighbor.", top_k=4))

**Concise Answer:**

* The affected neighbor's IP address is `<REDACTED:IPV4>`.
* Possible causes for BGP neighbor down with high CPU usage and memory utilization events:
	+ Misconfigured BGP neighbor statements
	+ Inconsistent or missing BGP router-id
	+ High CPU usage due to frequent BGP updates or neighbor changes
	+ Memory issues due to large BGP tables or neighbor connections
* Recommended first step: Verify BGP neighbor configuration and check for any inconsistencies or errors.

**Next Steps:**

1. Check BGP neighbor configuration for consistency and accuracy.
2. Verify BGP router-id and ensure it matches the Loopback0 interface IP address.
3. Monitor CPU and memory usage to identify any patterns or correlations with BGP neighbor changes.
4. Check BGP tables and neighbor connections for any issues or inconsistencies.

**Verification Steps:**

1. Check BGP neighbor status and verify that the affected neighbor is indeed down.
2. Check CPU and memory usage to confirm high utilizatio